In [1]:
!pip install nlpaug clean-text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.7.0-py3-none-any.whl size=171031 sha256=f42d5170f0c3a356aa33f9315aadd63dae48281fe0fd8b8ba95ef258c05e3064
  Stored in directory: /root/.cache/pip/wheels/bd/22/e5/b69726d5e1a19795ecd3b3e7464b16c0f1d019aa94ff1c8578
Successfully built emoji
  Attempting uninstall: emoji
    Found existing installation: emoji 2.14.1
    Uninstalling emoji-2.14.1:
      Successfully uninstalled emoji-2.14.1


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# Back translation imports
import nlpaug.augmenter.word as naw
from cleantext import clean

2025-09-25 06:26:27.333724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758781587.568122      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758781587.632604      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# CONFIG

In [3]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5 ############################CHAANGED
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
# Back translation augmenter with cleantext preprocessing
class BackTranslationAugmenter:
    def __init__(self, src_lang='en', intermediate_lang='fr', aug_p=0.3):
        self.aug_p = aug_p
        try:
            # Create back translation pipeline: en -> intermediate_lang -> en
            self.back_trans_aug = naw.BackTranslationAug(
                from_model_name=f'Helsinki-NLP/opus-mt-{src_lang}-{intermediate_lang}',
                to_model_name=f'Helsinki-NLP/opus-mt-{intermediate_lang}-{src_lang}',
                device='cuda' if torch.cuda.is_available() else 'cpu'
            )
            print(f"✅ Back translation augmenter initialized: {src_lang} -> {intermediate_lang} -> {src_lang}")
        except Exception as e:
            print(f"⚠️ Back translation setup failed: {e}")
            self.back_trans_aug = None
    
    def clean_text_for_translation(self, text):
        """Clean text using cleantext module for better translation"""
        if pd.isna(text) or text == '':
            return text, []
        
        # Store original URLs before cleaning
        import re
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        urls = re.findall(url_pattern, text)
        
        # Replace URLs with placeholders before cleaning
        clean_text = text
        for i, url in enumerate(urls):
            clean_text = clean_text.replace(url, f" URLTOKEN{i} ")
        
        # Use cleantext to clean the text
        clean_text = clean(clean_text,
                          fix_unicode=True,
                          to_ascii=False,
                          lower=False,
                          normalize_whitespace=True,
                          no_line_breaks=True,
                          strip_lines=True,
                          keep_two_line_breaks=False,
                          no_urls=False,  # We handle URLs separately
                          no_emails=False,
                          no_phone_numbers=False,
                          no_numbers=False,
                          no_digits=False,
                          no_currency_symbols=False,
                          no_punct=False,
                          lang="en")
        
        return clean_text, urls
    
    def restore_text_after_translation(self, text, urls):
        """Restore URLs after translation"""
        if pd.isna(text) or text == '':
            return text
        
        restored_text = text
        for i, url in enumerate(urls):
            restored_text = restored_text.replace(f"URLTOKEN{i}", url)
        
        return restored_text
    
    def augment_text(self, text):
        """Apply back translation to text"""
        if self.back_trans_aug is None or pd.isna(text) or text == '':
            return text
        
        try:
            # Clean text for translation
            clean_text, urls = self.clean_text_for_translation(text)
            
            # Apply back translation with probability
            if np.random.random() < self.aug_p and len(clean_text.split()) > 3:
                augmented = self.back_trans_aug.augment(clean_text)
                # Restore URLs
                if isinstance(augmented, list) and len(augmented) > 0:
                    # print(augmented)
                    augmented = augmented[0]
                augmented = self.restore_text_after_translation(augmented, urls)
                return augmented
            else:
                return text
                
        except Exception as e:
            print(f"Back translation failed for text: {text[:50]}... Error: {e}")
            return text
    
    def print_translation_examples(self, texts, n_examples=3):
        """Print examples of back translations"""
        print("=== Back Translation Examples ===")
        sample_texts = np.random.choice(texts, min(n_examples, len(texts)), replace=False)
        
        for i, original_text in enumerate(sample_texts, 1):
            if len(original_text.split()) > 3:
                translated = self.augment_text(original_text)
                print(f"\nExample {i}:")
                print(f"Original:   {original_text}")
                print(f"Translated: {translated}")
                print("-" * 80)

# Initialize back translation augmenter
print("Initializing back translation augmenter...")
back_translator = BackTranslationAugmenter(
    src_lang='en', 
    intermediate_lang='fr',  # German as intermediate language
    aug_p=.7  # 70% probability of augmentation
)

Initializing back translation augmenter...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ Back translation augmenter initialized: en -> fr -> en


In [7]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [8]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

def add_back_translation_data(texts, labels, back_translator, n_augmentations=1):
    """Add back-translated versions of the texts - only augmenting body part"""
    if back_translator.back_trans_aug is None:
        print("⚠️ Back translator not available, skipping back translation")
        return [], []
    
    bt_texts = []
    bt_labels = []
    
    print(f"Generating back translations for {len(texts)} texts (bodies only)...")
    for text, label in tqdm(zip(texts, labels), total=len(texts), desc="Back translating bodies"):
        for _ in range(n_augmentations):
            # Split the text into rule and body
            if ' [SEP] ' in text:
                rule_part, body_part = text.split(' [SEP] ', 1)
                
                # Only augment the body part if it's long enough
                if len(body_part.split()) > 5:
                    bt_body = back_translator.augment_text(body_part)
                    if bt_body != body_part:  # Only add if body actually changed
                        # Reconstruct with original rule + augmented body
                        bt_text = f"{rule_part} [SEP] {bt_body}"
                        bt_texts.append(bt_text)
                        bt_labels.append(label)
    
    print(f"✅ Generated {len(bt_texts)} back-translated examples (bodies only)")
    return bt_texts, bt_labels

In [9]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df (for positive/negative examples only)
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with positive/negative examples only (NO back translation yet)
augmented_texts = df.text.tolist() + augmented_train[0] + augmented_test[0]
augmented_labels = df.label.tolist() + augmented_train[1] + augmented_test[1]

# Create dataframe WITHOUT back translation first
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})

print(f'Before deduplication: {augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print(f'After deduplication: {augmented_df.shape}')

# Add rule parsing
augmented_df['rule'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map = {i: j for j, i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id'] = augmented_df.rule.str.lower().map(rule_map)

print("Base dataset ready (before back translation)")
augmented_df.head()

In [11]:
# test_df = pd.read_csv(test_path)

# # Get augmented data from both df and test_df
# augmented_train = add_data(df)
# augmented_test = add_data(test_df)

# # Combine original df with augmented data
# augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
# augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]


# # Create new augmented dataframe
# augmented_df = pd.DataFrame({
#     'text': augmented_texts,
#     'label': augmented_labels
# })
# print(f'Before:{augmented_df.shape}')
# augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
#     'text': 'first',  # Keep the original case of the first occurrence
#     'label': 'mean'   # Take mean of labels
# })
# print('After:',augmented_df.shape)
# augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
# augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

# rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
# augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

# augmented_df.head()

In [12]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [13]:
# print(df.head())
# print(df.columns.tolist())

In [14]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [15]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [16]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [17]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [18]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ', 'FOLD: ', fold, ' --------')
        
        # Split data into train and validation
        train_texts = augmented_df.iloc[tr_idx]['text'].tolist()
        train_labels = augmented_df.iloc[tr_idx]['label'].tolist()
        train_rule_ids = augmented_df.iloc[tr_idx]['rule_id'].tolist()
        
        val_texts = augmented_df.iloc[val_idx]['text'].tolist()
        val_labels = augmented_df.iloc[val_idx]['label'].tolist()
        val_rule_ids = augmented_df.iloc[val_idx]['rule_id'].tolist()
        
        # Apply back translation ONLY to training data (prevent data leakage)
        print(f"Adding back translation to training set only...")
        bt_texts, bt_labels = add_back_translation_data(
            train_texts, 
            train_labels, 
            back_translator, 
            n_augmentations=1
        )
        
        # Add back-translated examples to training set only
        if bt_texts:
            train_texts += bt_texts
            train_labels += bt_labels
            # Map back-translated texts to rule IDs
            bt_rule_ids = []
            for bt_text in bt_texts:
                rule_part = bt_text.split(' [SEP] ')[0]
                bt_rule_ids.append(rule_map[rule_part.lower()])
            train_rule_ids += bt_rule_ids
        
        print(f"Training set size after back translation: {len(train_texts)}")
        print(f"Validation set size: {len(val_texts)}")
        
        # Create datasets
        train_ds = JigsawDataset(train_texts, train_labels, train_rule_ids, tokenizer, MAX_LEN)
        val_ds = JigsawDataset(val_texts, val_labels, val_rule_ids, tokenizer, MAX_LEN)
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize classification model and load MLM pre-trained weights
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                
        print('Trainable Params: ', sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
        total_steps = EPOCHS * len(train_loader)
        warmup_steps = 0.1 * total_steps
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        best_auc = 0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss, val_preds = validate(model, val_loader)
            
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")

what to do at test time, dont take examples from test set into val.

In [20]:
all_truths=[]
all_rules=[]
all_preds=[]
folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
    
    wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
    wts= {k.replace('module.',''):v for k,v in wts.items()}
            
    model.load_state_dict(wts)
    model.eval()
    val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 

            tokenizer, MAX_LEN
        )
    all_truths.append(augmented_df.iloc[val_idx]['label'].apply(lambda x:x>=.5).astype('float'))
    all_rules.append(augmented_df.iloc[val_idx].rule)
        
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    _,_,val_preds= validate(model,val_loader)
            
    all_preds.append(pd.Series(val_preds))
            

preddf= pd.DataFrame(columns=['preds','truths','rule'])
preddf.preds=pd.concat(all_preds,ignore_index=True)
preddf.rule= pd.concat(all_rules,ignore_index=True)
preddf.truths= pd.concat(all_truths,ignore_index=True)

print(preddf.groupby('rule').apply(lambda group: roc_auc_score(group['truths'],group['preds'])))


rule
No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.    0.946199
No legal advice: Do not offer or request legal advice.                                                     0.889575
dtype: float64


/tmp/ipykernel_19/506043721.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(preddf.groupby('rule').apply(lambda group: roc_auc_score(group['truths'],group['preds'])))


In [21]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test),[0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [22]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv